# Stochastic Calculus, Step by Step · Lesson 8: Conditional expectation

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Averaging over a block · 3. Why it is a random variable · 4. The defining property: partial averages · 5. Four rules · 6. Averaging layer by layer · 7. The best prediction · 8. Conditioning on the price alone · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 8,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. Averaging over a block

After $n$ tosses you know which time-$n$ block the world is in. The **conditional expectation** $E[X \mid \mathcal F_n]$ is your best estimate of $X$ with that knowledge: the probability-weighted average of $X$ over the worlds in your block,

$$E[X \mid \mathcal F_n](\omega) = \frac{\sum_{v \in B(\omega)} X(v)\,P(v)}{P(B(\omega))},$$

where $B(\omega)$ is the time-$n$ block that contains $\omega$. For the share price, each toss multiplies the expected price by $pu + (1 - p)d = 1.02$, so $E[S_3 \mid \mathcal F_n] = 1.02^{\,3 - n}S_n$.

In [ ]:
def block(w, n):
    return [v for v in worlds if v[:n] == w[:n]]


def cond_exp(X, n, q=p):
    # E[X | F_n] as a dict world -> value: average X over the time-n block, with world weights prob(., q).
    res = {}
    for w in worlds:
        b_ = block(w, n)
        res[w] = sum(X(v) * prob(v, q) for v in b_) / sum(prob(v, q) for v in b_)
    return res


S3 = lambda w: price(w, 3)
m = p * u + (1 - p) * d                       # expected growth per toss under p
ce = {n: cond_exp(S3, n) for n in range(steps + 1)}
for n in range(steps + 1):
    assert all(abs(ce[n][w] - m ** (steps - n) * price(w, n)) < 1e-9 for w in worlds)
out["ce"] = {"growth": m, "growth2": m**2, "growth3": m**3,
             "rows": [{"world": w, **{f"n{n}": ce[n][w] for n in range(steps + 1)}} for w in worlds],
             "ES3": ce[0]["HHH"], "H": ce[1]["HHH"], "T": ce[1]["TTT"],
             "HH": ce[2]["HHH"], "HT": ce[2]["HTH"], "TH": ce[2]["THH"], "TT": ce[2]["TTT"]}
pd.DataFrame(out["ce"]["rows"]).set_index("world").round(4)

## 3. Why it is a random variable

Before any toss, $E[S_3 \mid \mathcal F_1]$ is not a number: it will be $1.02^2 \times 110 = 114.444$ if the first toss is heads and $1.02^2 \times 90 = 93.636$ if it is tails. So it is a random variable, known at time 1, with its own distribution. Its average is the plain expectation $E[S_3]$.

In [ ]:
ce1 = {"values": [out["ce"]["H"], out["ce"]["T"]], "probs": [p, 1 - p]}
ce1["mean"] = p * out["ce"]["H"] + (1 - p) * out["ce"]["T"]
assert abs(ce1["mean"] - out["ce"]["ES3"]) < 1e-9
out["ce1"] = ce1
ce1

## 4. The defining property: partial averages

$E[X \mid \mathcal G]$ is the only $\mathcal G$-measurable random variable with the same average as $X$ over every event in $\mathcal G$:

$$E\big[E[X \mid \mathcal G]\,\mathbf 1_A\big] = E[X\,\mathbf 1_A] \qquad \text{for every } A \in \mathcal G .$$

Check it for the call payoff $X = \max(S_3 - 105, 0)$ and all 16 events of $\mathcal F_2$.

In [ ]:
call = lambda w: max(price(w, 3) - K, 0.0)
E = lambda f: sum(f(w) * prob(w) for w in worlds)
ce_call2 = cond_exp(call, 2)
bl2 = [block(w, 2) for w in ("HHH", "HTH", "THH", "TTH")]
worst = 0.0
for pick in itertools.product([0, 1], repeat=4):
    A = {w for b_, k_ in zip(bl2, pick) if k_ for w in b_}
    lhs = E(lambda w: ce_call2[w] * (w in A))
    rhs = E(lambda w: call(w) * (w in A))
    worst = max(worst, abs(lhs - rhs))
A_h = {w for w in worlds if w[0] == "H"}
out["partial"] = {"max_gap": worst, "n_events": 16,
                  "A_h_lhs": E(lambda w: ce_call2[w] * (w in A_h)), "A_h_rhs": E(lambda w: call(w) * (w in A_h))}
out["partial"]

## 5. Four rules

1. **Linearity:** $E[aX + bY \mid \mathcal G] = aE[X \mid \mathcal G] + bE[Y \mid \mathcal G]$.
2. **Taking out what is known:** if $Y$ is $\mathcal G$-measurable, $E[YX \mid \mathcal G] = Y\,E[X \mid \mathcal G]$.
3. **Independence:** if $X$ is independent of $\mathcal G$, $E[X \mid \mathcal G] = E[X]$.
4. **Tower:** if $\mathcal H \subseteq \mathcal G$ (less information), $E\big[E[X \mid \mathcal G] \mid \mathcal H\big] = E[X \mid \mathcal H]$. In particular $E\big[E[X \mid \mathcal G]\big] = E[X]$.

Each one checked numerically on the tree:

In [ ]:
S1 = lambda w: price(w, 1); S2 = lambda w: price(w, 2)
gap = lambda a, b_: max(abs(a[w] - b_[w]) for w in worlds)
lin = gap(cond_exp(lambda w: 2 * S3(w) - 3 * call(w), 1), {w: 2 * ce[1][w] - 3 * cond_exp(call, 1)[w] for w in worlds})
known_ = gap(cond_exp(lambda w: S1(w) * S3(w), 1), {w: S1(w) * ce[1][w] for w in worlds})
ret3 = lambda w: price(w, 3) / price(w, 2)            # the third step's growth: independent of the first two tosses
indep = max(abs(v - m) for v in cond_exp(ret3, 2).values())
inner = cond_exp(call, 2)
tower = gap(cond_exp(lambda w: inner[w], 1), cond_exp(call, 1))
out["rules"] = {"linearity": lin, "known": known_, "independence": indep, "tower": tower,
                "E_S1S3": E(lambda w: S1(w) * S3(w)), "E_S1sq_m2": m**2 * E(lambda w: S1(w) ** 2)}
assert max(lin, known_, indep, tower) < 1e-9
out["rules"]

## 6. Averaging layer by layer

The tower rule gives a way to compute $E[X]$ for a payoff at time 3: average the payoff over the last toss at each time-2 node, then average those over the second toss, then over the first. On a recombining tree only the number of heads matters, so each layer has one value per node. (This is the real-world average payoff, not a price: prices need different weights and discounting, from Lesson 9.)

In [ ]:
nodes = []
vals = {(steps, k_): max(S0_ * u**k_ * d ** (steps - k_) - K, 0.0) for k_ in range(steps + 1)}
for n in range(steps - 1, -1, -1):
    for k_ in range(n + 1):
        vals[(n, k_)] = p * vals[(n + 1, k_ + 1)] + (1 - p) * vals[(n + 1, k_)]
for n in range(steps + 1):
    for k_ in range(n + 1):
        nodes.append({"t": n, "k": k_, "price": S0_ * u**k_ * d ** (n - k_), "value": vals[(n, k_)]})
out["layers"] = {"nodes": nodes, "root": vals[(0, 0)], "direct": E(call)}
assert abs(out["layers"]["root"] - out["layers"]["direct"]) < 1e-9
pd.DataFrame(nodes).round(4)

## 7. The best prediction

Among all guesses $Y$ that use only time-$n$ information, $Y = E[X \mid \mathcal F_n]$ has the smallest mean squared error $E[(X - Y)^2]$. Try the guesses $Y = c\,S_1$ for a range of $c$: the error is smallest at $c = 1.02^2$, the conditional expectation.

In [ ]:
cs = np.linspace(0.95, 1.12, 1701)
mse = [E(lambda w: (S3(w) - c * S1(w)) ** 2) for c in cs]
out["best"] = {"c_best": float(cs[int(np.argmin(mse))]), "c_theory": m**2,
               "mse_ce": E(lambda w: (S3(w) - ce[1][w]) ** 2), "mse_mean": E(lambda w: (S3(w) - ce[0][w]) ** 2),
               "mse_now": E(lambda w: (S3(w) - S1(w)) ** 2)}
out["best"]["fig"] = {"c": cs[::10].tolist(), "mse": mse[::10],
                      "ticks": [{"x": x, "label": f"{x:.2f}"} for x in (0.96, 1.0, 1.04, 1.08, 1.12)]}
out["best"]

## 8. Conditioning on the price alone

$\sigma(S_2)$ holds less information than $\mathcal F_2$: HT and TH give the same price, 99. For $X = S_3$ it makes no difference, because the future of the price depends only on today's price. That is the **Markov property**, the subject of Lesson 11. (Exercise 8.8 shows a payoff for which it does matter.)

In [ ]:
def cond_exp_on(X, key):
    # E[X | sigma(key)]: average X over worlds that share the value of key.
    res = {}
    for w in worlds:
        b_ = [v for v in worlds if abs(key(v) - key(w)) < 1e-9]
        res[w] = sum(X(v) * prob(v) for v in b_) / sum(prob(v) for v in b_)
    return res


out["markov"] = {"gap_S3": gap(cond_exp_on(S3, S2), ce[2])}
out["markov"]

## 9. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** Compute $E[S_2 \mid \mathcal F_1]$ on each time-1 block directly from the definition (average over the block), and check that it equals $1.02\,S_1$.

**Exercise 2.** Show that $E[S_{n+1} \mid \mathcal F_n] = \big(pu + (1 - p)d\big)S_n$ for any $n$. For which $p$ is the price expected to stay where it is?

**Exercise 3.** Use *taking out what is known* to find $E[S_1S_3 \mid \mathcal F_1]$, then use the tower rule to find $E[S_1S_3]$.

**Exercise 4.** Let $G = S_3/S_2$, the third step's growth factor. Show that $G$ is independent of $\mathcal F_2$ and compute $E[G \mid \mathcal F_2]$. Why is $E[S_3 \mid \mathcal F_2] = S_2\,E[G]$?

**Exercise 5.** Verify the tower rule by hand at the time-1 block H: compute $E[S_3 \mid \mathcal F_2]$ on the blocks HH and HT, average them, and compare with $E[S_3 \mid \mathcal F_1]$ on H.

**Exercise 6.** For $X = \max(S_3 - 105, 0)$ and $A = \{\text{first toss H}\}$, check the partial-averaging property $E\big[E[X \mid \mathcal F_1]\mathbf 1_A\big] = E[X\mathbf 1_A]$ by computing both sides.

**Exercise 7.** (Code) Write a function `cond_exp(X, n)` for the three-toss tree. Draw a random payoff $X$ (eight random numbers, one per world) and check the tower rule $E\big[E[X \mid \mathcal F_2] \mid \mathcal F_1\big] = E[X \mid \mathcal F_1]$.

**Exercise 8.** (Code) Let $M_3 = \max(S_0, S_1, S_2, S_3)$. Compute $E[M_3 \mid \mathcal F_2]$ and $E[M_3 \mid S_2]$ in every world. Where do they differ, and why?

## 10. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")